### Verify GPU

In [2]:
import torch

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device  : {torch.cuda.get_device_name(0)}")

PyTorch : 2.11.0+cu128
GPU     : True
Device  : Tesla T4


### Mount Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT = '/content/drive/MyDrive/L4_Research_Resources/SP_GaRT'
os.makedirs(f'{DRIVE_PROJECT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/data',        exist_ok=True)
print("Drive mounted. Folders ready.")

Mounted at /content/drive
Drive mounted. Folders ready.


### Clone repository

In [ ]:
if not os.path.exists('/content/SP_GaRT'):
    !git clone https://github.com/GayuniBas2001/SP-GaRT_Spatially_Pruned_Graph_Transformer.git /content/SP_GaRT
    print("Cloned.")
else:
    !cd /content/SP_GaRT && git pull
    print("Updated.")

Cloning into '/content/SP_GaRT'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 87 (delta 27), reused 68 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 369.85 KiB | 7.55 MiB/s, done.
Resolving deltas: 100% (27/27), done.
Cloned.


In [ ]:


%cd /content/SP_GaRT

/content/SP_GaRT


In [ ]:
import sys
sys.path.insert(0, '/content/SP_GaRT')
print("Path set.")

Path set.


### Copy data

In [ ]:
DATA_LOCAL = '/content/drive/MyDrive/L4_Research_Resources/SP_GaRT/data/data_3d_h36m.npz'
DATA_DRIVE = f'{DRIVE_PROJECT}/data/data_3d_h36m.npz'

In [ ]:
if not os.path.exists(DATA_LOCAL):
    !cp "{DATA_DRIVE}" "{DATA_LOCAL}"
    print("Data copied.")
else:
    print("Data already present.")

Data already present.


In [ ]:
print(f"File size: {os.path.getsize(DATA_LOCAL)/1e6:.1f}MB")

File size: 153.2MB


### Install and import

In [ ]:
!pip install tensorboard -q

In [ ]:
from data.h36m_dataset import build_dataloaders
from utils.metrics import mpjpe_at_horizons, gravity_violation_rate, bone_length_error
from utils.trainer import train_model, evaluate_model, print_results, measure_inference_time

print("All imports successful.")

All imports successful.


### Load data

In [ ]:
device = torch.device('cuda')

train_loader, test_loader = build_dataloaders(
    DATA_LOCAL, batch_size=32,
    t_obs=10, t_pred=25,
    train_stride=5, test_stride=1
)

batch = next(iter(train_loader))
print(f"Observed : {batch['observed'].shape}")
print(f"Future   : {batch['future'].shape}")

[H36MDataset] 38045 windows | subjects=['S1', 'S5', 'S6', 'S7', 'S8'] | t_obs=10 t_pred=25 | stride=5 | 25Hz
[H36MDataset] 65927 windows | subjects=['S9', 'S11'] | t_obs=10 t_pred=25 | stride=1 | 25Hz
Observed : torch.Size([32, 10, 17, 3])
Future   : torch.Size([32, 25, 17, 3])


In [ ]:
!git pull

Already up to date.


# Training

### Monitor training

In [ ]:
DRIVE_PROJECT = '/content/drive/MyDrive/L4_Research_Resources/SP_GaRT'
SAVE_DIR = f'{DRIVE_PROJECT}/checkpoints'

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_PROJECT}/runs
# LOG_DIR = '/content/drive/MyDrive/L4 Research Resources/SP_GaRT/runs'
# %tensorboard --logdir "{LOG_DIR}"

### Import and verify M1 : Vanila Transformer

In [ ]:
from models.vanilla_transformer import VanillaTransformer

model_M1 = VanillaTransformer(
    J=17, D=3,
    d_model=256, n_heads=4,
    n_enc_layers=2, n_dec_layers=2,
    d_ff=512, dropout=0.1,
    t_obs=10, t_pred=25
).to(device)

print(f"Parameters : {model_M1.count_parameters():,}")

# Test forward pass
dummy = torch.randn(32, 10, 17, 3).to(device)
out   = model_M1(dummy)
print(f"Input      : {dummy.shape}")
print(f"Output     : {out.shape}")
assert out.shape == (32, 25, 17, 3)
print("✓ Forward pass correct")

Parameters : 2,668,595
Input      : torch.Size([32, 10, 17, 3])
Output     : torch.Size([32, 25, 17, 3])
✓ Forward pass correct


### Train M1

In [ ]:
# SAVE_DIR = f'{DRIVE_PROJECT}/checkpoints'

# trained_M1 = train_model(
#     model           = model_M1,
#     train_loader    = train_loader,
#     test_loader     = test_loader,
#     n_epochs        = 50,
#     lr              = 1e-3,
#     device          = device,
#     experiment_name = 'M1_vanilla_transformer',
#     model_config    = {
#         'J': 17, 'D': 3,
#         'd_model': 256, 'n_heads': 4,
#         'n_enc_layers': 2, 'n_dec_layers': 2,
#         'd_ff': 512, 't_obs': 10, 't_pred': 25
#     },
#     eval_every  = 5,
#     save_dir    = SAVE_DIR,
#     log_dir     = f'{DRIVE_PROJECT}/runs',
#     resume      = True
# )

### Import and verify M2 : Dense graph - all joints equal weight (Fixed anatomical bias)

In [ ]:
# ── Force re-import since Python caches modules ─────────────
import importlib
import models.graph_transformer as gt_module
importlib.reload(gt_module)

<module 'models.graph_transformer' from '/content/SP_GaRT/models/graph_transformer.py'>

In [ ]:
from models.graph_transformer import DenseGraphTransformer

# ── Verify the class matches the verified pipeline ──────────
model_M2 = DenseGraphTransformer(
    J=17, D=3,
    d_model=256, n_heads=4,
    n_st_layers=2, d_ff=512,
    dropout=0.1, t_obs=10, t_pred=25
).to(device)

print(f"M1 parameters : {model_M1.count_parameters():,}")
print(f"M2 parameters : {model_M2.count_parameters():,}")
print(f"M2 should have more parameters than M1")
print()

# Test forward pass with real data
real_sample = next(iter(test_loader))
obs_sample  = real_sample['observed'][:4].to(device)
fut_sample  = real_sample['future'][:4].to(device)

with torch.no_grad():
    pred_M2 = model_M2(obs_sample)

print(f"Input  : {obs_sample.shape}")
print(f"Output : {pred_M2.shape}")
assert pred_M2.shape == (4, 25, 17, 3)
print()

# Confirm not collapsed
var_time  = pred_M2.var(dim=1).mean().item()
var_batch = pred_M2.var(dim=0).mean().item()
print(f"Variance across time  : {var_time:.6f}  (must be > 0.001)")
print(f"Variance across batch : {var_batch:.6f}  (must be > 0.001)")
assert var_time  > 0.001, "Output collapsed across time"
assert var_batch > 0.001, "Output collapsed across batch"

# Adjacency matrix loaded correctly
print(f"Adjacency matrix on device : {model_M2.adj.device}")
print(f"Adjacency connections      : {int(model_M2.adj.sum().item())}")
assert int(model_M2.adj.sum().item()) == 49

print()
print("✓ DenseGraphTransformer class verified")
print("✓ Ready to train M2")

M1 parameters : 2,668,595
M2 parameters : 4,827,195
M2 should have more parameters than M1

Input  : torch.Size([4, 10, 17, 3])
Output : torch.Size([4, 25, 17, 3])

Variance across time  : 0.626073  (must be > 0.001)
Variance across batch : 0.387702  (must be > 0.001)
Adjacency matrix on device : cuda:0
Adjacency connections      : 49

✓ DenseGraphTransformer class verified
✓ Ready to train M2


### Train M2

In [ ]:
# # ── Train M2 ────────────────────────────────────────────────
# model_M2 = DenseGraphTransformer(
#     J=17, D=3,
#     d_model=256, n_heads=4,
#     n_st_layers=2, d_ff=512,
#     dropout=0.1, t_obs=10, t_pred=25
# ).to(device)

# trained_M2 = train_model(
#     model           = model_M2,
#     train_loader    = train_loader,
#     test_loader     = test_loader,
#     n_epochs        = 50,
#     lr              = 1e-4,
#     device          = device,
#     experiment_name = 'M2_dense_graph_transformer',
#     model_config    = {
#         'J': 17, 'D': 3,
#         'd_model': 256, 'n_heads': 4,
#         'n_st_layers': 2, 'd_ff': 512,
#         't_obs': 10, 't_pred': 25
#     },
#     eval_every = 5,
#     save_dir   = SAVE_DIR,
#     log_dir    = f'{DRIVE_PROJECT}/runs',
#     resume     = False   # fresh start — old checkpoints deleted
# )

### Import and verify M4

**Create the Combined Loss Function**

In [ ]:
# ── M4 Loss Function ─────────────────────────────────────────
import torch
import torch.nn as nn
from utils.metrics import gravity_consistency_loss

class SPGaRTLoss(nn.Module):
    """
    Combined loss for SP-GaRT (M4).

    L_total = L_recon + lambda * L_gravity

    L_recon  : MSE between predicted and ground truth poses
    L_gravity: Differentiable gravity-consistency penalty
               Applied only to standing frames (root Z > 0.70m)
               Uses relu to penalise BoS violations

    Args:
        lambda_gravity: weight for gravity loss (default 0.01)
    """
    def __init__(self, lambda_gravity=0.01):
        super().__init__()
        self.lambda_gravity = lambda_gravity
        self.mse = nn.MSELoss()

    def forward(self, predicted, future, observed):
        """
        Args:
            predicted : (B, T_pred, J, 3) — model output
            future    : (B, T_pred, J, 3) — ground truth
            observed  : (B, T_obs,  J, 3) — input sequence
                        needed to detect standing posture
        Returns:
            total_loss, recon_loss, gravity_loss
        """
        # Primary accuracy signal
        l_recon = self.mse(predicted, future)

        # Physics regulariser — differentiable via relu
        l_gravity = gravity_consistency_loss(
            predicted, observed
        )

        total = l_recon + self.lambda_gravity * l_gravity

        return total, l_recon, l_gravity


# ── Verify loss function ──────────────────────────────────────
loss_fn_M4 = SPGaRTLoss(lambda_gravity=0.01)

sample     = next(iter(test_loader))
obs_v      = sample['observed'][:4].to(device)
fut_v      = sample['future'][:4].to(device)

# Simulate model output
with torch.no_grad():
    pred_v = model_M2(obs_v)

total, recon, gravity = loss_fn_M4(pred_v, fut_v, obs_v)

print("=" * 50)
print("M4 LOSS FUNCTION VERIFICATION")
print("=" * 50)
print(f"L_recon   : {recon.item():.6f}")
print(f"L_gravity : {gravity.item():.6f}")
print(f"L_total   : {total.item():.6f}")
print(f"Lambda    : {loss_fn_M4.lambda_gravity}")
print()

# Verify gravity loss is smaller than recon loss
# (lambda=0.01 ensures gravity does not overpower recon)
ratio = (loss_fn_M4.lambda_gravity *
         gravity.item()) / recon.item()
print(f"Gravity contribution to total: {ratio*100:.2f}%")
print(f"  Should be small (< 10%) so recon remains dominant")
print()

# Verify gradient flows
pred_grad = model_M2(obs_v)
t, r, g   = loss_fn_M4(pred_grad, fut_v, obs_v)
t.backward()
print(f"✓ Gradients flow through combined loss")
print("=" * 50)
print("M4 Loss verified — ready to train")
print("=" * 50)

M4 LOSS FUNCTION VERIFICATION
L_recon   : 2.184416
L_gravity : 0.906653
L_total   : 2.193483
Lambda    : 0.01

Gravity contribution to total: 0.42%
  Should be small (< 10%) so recon remains dominant

✓ Gradients flow through combined loss
M4 Loss verified — ready to train


**The Training Loop Needs One Modification**

Note : The existing train_model function passes only (predicted, future) to the loss. M4's loss also needs observed for the standing detection.

Before training:

In [ ]:
# ── Custom training loop for M4 ──────────────────────────────
# Extends train_model to pass 'observed' to the loss function

def train_model_M4(model, train_loader, test_loader,
                   n_epochs, lr, device,
                   experiment_name,
                   lambda_gravity=0.01,
                   eval_every=5,
                   save_dir='checkpoints',
                   log_dir='runs',
                   resume=True):
    """
    Training loop for M4 SP-GaRT.
    Same as train_model but passes observed to loss function
    so gravity_consistency_loss can detect standing frames.
    """
    import os
    from torch.utils.tensorboard import SummaryWriter

    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(f'{log_dir}/{experiment_name}', exist_ok=True)

    writer    = SummaryWriter(f'{log_dir}/{experiment_name}')
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[30, 60, 80], gamma=0.5
    )
    loss_fn   = SPGaRTLoss(lambda_gravity=lambda_gravity)

    start_epoch = 1
    best_mpjpe  = float('inf')
    latest_path = f'{save_dir}/{experiment_name}_latest.pth'
    best_path   = f'{save_dir}/{experiment_name}_best.pth'

    if resume and os.path.exists(latest_path):
        print(f"Resuming: {latest_path}")
        ckpt = torch.load(latest_path, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        scheduler.load_state_dict(ckpt['scheduler_state'])
        start_epoch = ckpt['epoch'] + 1
        best_mpjpe  = ckpt.get('best_mpjpe', float('inf'))
        print(f"  Resumed at epoch {start_epoch} | "
              f"Best MPJPE@560ms: {best_mpjpe:.1f}mm")
    else:
        print(f"Starting: {experiment_name}")

    for epoch in range(start_epoch, n_epochs + 1):
        # ── Training ──────────────────────────────────────────
        model.train()
        total_loss   = 0.0
        total_recon  = 0.0
        total_grav   = 0.0
        n_batches    = 0

        for batch in train_loader:
            obs  = batch['observed'].to(device)
            fut  = batch['future'].to(device)

            optimizer.zero_grad()
            pred = model(obs)

            # Key difference from M2:
            # pass obs so gravity loss can detect standing
            loss, l_recon, l_grav = loss_fn(pred, fut, obs)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0
            )
            optimizer.step()

            total_loss  += loss.item()
            total_recon += l_recon.item()
            total_grav  += l_grav.item()
            n_batches   += 1

        avg_loss  = total_loss  / n_batches
        avg_recon = total_recon / n_batches
        avg_grav  = total_grav  / n_batches
        scheduler.step()

        # ── Logging ───────────────────────────────────────────
        writer.add_scalar('Loss/total',   avg_loss,  epoch)
        writer.add_scalar('Loss/recon',   avg_recon, epoch)
        writer.add_scalar('Loss/gravity', avg_grav,  epoch)
        writer.add_scalar('LR',
            optimizer.param_groups[0]['lr'], epoch)

        # ── Save latest checkpoint every epoch ────────────────
        torch.save({
            'epoch':           epoch,
            'model_state':     model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'best_mpjpe':      best_mpjpe,
            'train_loss':      avg_loss,
        }, latest_path)

        # ── Evaluate periodically ─────────────────────────────
        if epoch % eval_every == 0:
            from utils.trainer import evaluate_model
            results   = evaluate_model(
                model, test_loader, device, n_batches=50
            )
            mpjpe_560 = results['mpjpe'][560]
            gvr       = results['gvr']

            writer.add_scalar('MPJPE/560ms', mpjpe_560, epoch)
            if gvr is not None:
                writer.add_scalar('GVR', gvr, epoch)

            gvr_str = f"{gvr:.4f}" if gvr is not None else "N/A"
            print(f"Epoch {epoch:>3}/{n_epochs} | "
                  f"loss: {avg_loss:.5f} "
                  f"(recon: {avg_recon:.5f}, "
                  f"grav: {avg_grav:.5f}) | "
                  f"MPJPE@560ms: {mpjpe_560:.1f}mm | "
                  f"GVR: {gvr_str}")

            if mpjpe_560 < best_mpjpe:
                best_mpjpe = mpjpe_560
                torch.save({
                    'epoch':       epoch,
                    'model_state': model.state_dict(),
                    'results':     results,
                    'best_mpjpe':  best_mpjpe,
                }, best_path)
                print(f"  ✓ Best saved: "
                      f"MPJPE@560ms={best_mpjpe:.1f}mm")

        elif epoch % 5 == 0:
            print(f"Epoch {epoch:>3}/{n_epochs} | "
                  f"loss: {avg_loss:.5f} "
                  f"(recon: {avg_recon:.5f}, "
                  f"grav: {avg_grav:.5f})")

    writer.close()
    print(f"\nDone. Best MPJPE@560ms: {best_mpjpe:.1f}mm")
    print(f"Best checkpoint: {best_path}")
    return model

### Train M4

In [ ]:
# # ── Train M4 SP-GaRT ─────────────────────────────────────────
# from models.graph_transformer import DenseGraphTransformer

# model_M4 = DenseGraphTransformer(
#     J=17, D=3,
#     d_model=256, n_heads=4,
#     n_st_layers=2, d_ff=512,
#     dropout=0.1, t_obs=10, t_pred=25
# ).to(device)

# trained_M4 = train_model_M4(
#     model           = model_M4,
#     train_loader    = train_loader,
#     test_loader     = test_loader,
#     n_epochs        = 50,
#     lr              = 1e-4,
#     device          = device,
#     experiment_name = 'M4_SP_GaRT',
#     lambda_gravity  = 0.01,
#     eval_every      = 5,
#     save_dir        = SAVE_DIR,
#     log_dir         = f'{DRIVE_PROJECT}/runs',
#     resume          = False
# )

**M4 Training Log Interpretation**

- Loss components behaved correctly:
```
Epoch 5  : recon=0.01442, grav=0.00484
Epoch 25 : recon=0.00696, grav=0.00221
Epoch 50 : recon=0.00467, grav=0.00169
```

Both recon and gravity losses decreased steadily throughout training. The gravity loss contribution to total loss is approximately 3% (0.00484 × 0.01 / 0.01447 at epoch 5) — small enough that it does not overpower the reconstruction signal. This is exactly what lambda=0.01 was designed to achieve.

- GVR behaviour is interesting and important:
```
Epoch 5  : 0.0000
Epoch 20 : 0.0002
Epoch 25 : 0.0077  ← temporary spike
Epoch 35 : 0.0001
Epoch 50 : 0.0000
```

GVR fluctuates during training rather than monotonically decreasing. This is because GVR is evaluated on 50 random batches which may or may not contain standing frames — the variation reflects batch sampling rather than the model getting worse. The overall trend is near-zero GVR, which is good.


**Three-Model Comparison**

```
                  M1          M2          M4
MPJPE@560ms  : 142.4mm    105.8mm    104.6mm
Best epoch   :  50          40          35
Final loss   : ~0.006      0.00460    0.00469
GVR (train)  : 0.0548     0.000275   0.0000
```

M1 → M2: +26.4mm improvement - graph structure clearly helps

M2 → M4: +1.2mm improvement** - gravity loss adds marginal MPJPE benefit

The MPJPE improvement from M2 to M4 is small (1.2mm) which is expected — the gravity loss was never designed to improve MPJPE, it was designed to improve physical plausibility. The key comparison is GVR.

### Import and verify M3a : Heuristic pruning - gaze + proximity rule

In [ ]:
# Reimport
import importlib
import models.pruned_graph_transformer as pgt_module
importlib.reload(pgt_module)
from models.pruned_graph_transformer import PrunedGraphTransformer

# Verify
model_M3 = PrunedGraphTransformer(
    J=17, D=3, d_model=256, n_heads=4,
    n_st_layers=2, d_ff=512,
    dropout=0.1, t_obs=10, t_pred=25
).to(device)

print(f"M2 parameters : {model_M2.count_parameters():,}")
print(f"M3 parameters : {model_M3.count_parameters():,}")
print("M3 should be close to M2 (same architecture + small pruning overhead)")

# Test forward pass
sample  = next(iter(test_loader))
obs_t   = sample['observed'][:4].to(device)
fut_t   = sample['future'][:4].to(device)

with torch.no_grad():
    pred_M3 = model_M3(obs_t)

assert pred_M3.shape == (4, 25, 17, 3)
print(f"✓ Output shape: {pred_M3.shape}")

# Confirm pruning mask is computed and not collapsed
var = pred_M3.var().item()
print(f"✓ Output variance: {var:.4f}  (must be > 0.001)")
assert var > 0.001
print("✓ M3 verified — ready to train")

M2 parameters : 4,827,195
M3 parameters : 4,827,195
M3 should be close to M2 (same architecture + small pruning overhead)
✓ Output shape: torch.Size([4, 25, 17, 3])
✓ Output variance: 1.5937  (must be > 0.001)
✓ M3 verified — ready to train


### Train M3a

In [ ]:
# model_M3 = PrunedGraphTransformer(
#     J=17, D=3, d_model=256, n_heads=4,
#     n_st_layers=2, d_ff=512,
#     dropout=0.1, t_obs=10, t_pred=25
# ).to(device)

# trained_M3 = train_model(
#     model           = model_M3,
#     train_loader    = train_loader,
#     test_loader     = test_loader,
#     n_epochs        = 50,
#     lr              = 1e-4,
#     device          = device,
#     experiment_name = 'M3_pruned_graph_transformer',
#     model_config    = {
#         'J': 17, 'D': 3, 'd_model': 256, 'n_heads': 4,
#         'n_st_layers': 2, 't_obs': 10, 't_pred': 25
#     },
#     eval_every = 5,
#     save_dir   = SAVE_DIR,
#     log_dir    = f'{DRIVE_PROJECT}/runs',
#     resume     = False
# )

### Import and verify M3b : Learned pruning - neural relevance network

In [ ]:
import importlib
import models.pruned_graph_transformer_v2 as m3b_module
importlib.reload(m3b_module)
from models.pruned_graph_transformer_v2 import (
    LearnedPrunedGraphTransformer
)

# Verify
model_M3b = LearnedPrunedGraphTransformer(
    J=17, D=3, d_model=256, n_heads=4,
    n_st_layers=2, d_ff=512,
    dropout=0.1, t_obs=10, t_pred=25
).to(device)

print(f"M2  parameters : {model_M2.count_parameters():,}")
print(f"M3a parameters : {model_M3.count_parameters():,}")
print(f"M3b parameters : {model_M3b.count_parameters():,}")
print("M3b should be slightly larger than M3a due to relevance network")

# Test
sample   = next(iter(test_loader))
obs_test = sample['observed'][:4].to(device)
with torch.no_grad():
    pred_M3b = model_M3b(obs_test)

assert pred_M3b.shape == (4, 25, 17, 3)
print(f"✓ Output shape: {pred_M3b.shape}")
print(f"✓ Variance: {pred_M3b.var().item():.4f}")
print("✓ M3b ready to train")

M2  parameters : 4,827,195
M3a parameters : 4,827,195
M3b parameters : 4,830,652
M3b should be slightly larger than M3a due to relevance network
✓ Output shape: torch.Size([4, 25, 17, 3])
✓ Variance: 1.7931
✓ M3b ready to train


### Train M3b

In [ ]:
# model_M3b = LearnedPrunedGraphTransformer(
#     J=17, D=3, d_model=256, n_heads=4,
#     n_st_layers=2, d_ff=512,
#     dropout=0.1, t_obs=10, t_pred=25
# ).to(device)

# trained_M3b = train_model(
#     model           = model_M3b,
#     train_loader    = train_loader,
#     test_loader     = test_loader,
#     n_epochs        = 50,
#     lr              = 1e-4,
#     device          = device,
#     experiment_name = 'M3b_learned_pruned_transformer',
#     model_config    = {
#         'type': 'LearnedPrunedGraphTransformer',
#         'J': 17, 'D': 3, 'd_model': 256, 'n_heads': 4,
#         'n_st_layers': 2, 't_obs': 10, 't_pred': 25
#     },
#     eval_every = 5,
#     save_dir   = SAVE_DIR,
#     log_dir    = f'{DRIVE_PROJECT}/runs',
#     resume     = False
# )

### Import and verify M5 : True SP-GaRT - M3a + Gravity Loss

In [1]:
import importlib
import models.pruned_graph_transformer as m5_module
importlib.reload(m5_module)
from models.pruned_graph_transformer import PrunedGraphTransformer

# Verify
model_M5 = PrunedGraphTransformer(
    J=17, D=3, d_model=256, n_heads=4,
    n_st_layers=2, d_ff=512,
    dropout=0.1, t_obs=10, t_pred=25
).to(device)

print(f"M2  parameters : {model_M2.count_parameters():,}")
print(f"M3a parameters : {model_M3.count_parameters():,}")
print(f"M5  parameters : {model_M5.count_parameters():,}")
print("M5 should have identical parameters to M3a")
print("(same architecture — only the training loss differs)")

# Test
sample   = next(iter(test_loader))
obs_test = sample['observed'][:4].to(device)
with torch.no_grad():
    pred_M5 = model_M5(obs_test)

assert pred_M5.shape == (4, 25, 17, 3), \
    f"Wrong shape: {pred_M5.shape}"
print(f"✓ Output shape: {pred_M5.shape}")
print(f"✓ Variance: {pred_M5.var().item():.4f}")
print("✓ M5 verified — ready to train with gravity loss")

ModuleNotFoundError: No module named 'models'

# Train M5

In [ ]:
# ── True SP-GaRT: M3a architecture + Gravity-Consistency Loss ─
from models.pruned_graph_transformer import PrunedGraphTransformer

model_M5 = PrunedGraphTransformer(
    J=17, D=3, d_model=256, n_heads=4,
    n_st_layers=2, d_ff=512,
    dropout=0.1, t_obs=10, t_pred=25
).to(device)

trained_M5 = train_model_M4(   # reuse M4 training loop with gravity loss
    model           = model_M5,
    train_loader    = train_loader,
    test_loader     = test_loader,
    n_epochs        = 50,
    lr              = 1e-4,
    device          = device,
    experiment_name = 'M5_true_SP_GaRT',
    lambda_gravity  = 0.01,
    eval_every      = 5,
    save_dir        = SAVE_DIR,
    log_dir         = f'{DRIVE_PROJECT}/runs',
    resume          = False
)

Starting: M5_true_SP_GaRT
Epoch   5/50 | loss: 0.01451 (recon: 0.01446, grav: 0.00450) | MPJPE@560ms: 144.6mm | GVR: 0.0000
  ✓ Best saved: MPJPE@560ms=144.6mm
Epoch  10/50 | loss: 0.01065 (recon: 0.01062, grav: 0.00319) | MPJPE@560ms: 120.6mm | GVR: 0.0000
  ✓ Best saved: MPJPE@560ms=120.6mm
Epoch  15/50 | loss: 0.00882 (recon: 0.00880, grav: 0.00269) | MPJPE@560ms: 127.9mm | GVR: 0.0000
Epoch  20/50 | loss: 0.00768 (recon: 0.00765, grav: 0.00242) | MPJPE@560ms: 112.6mm | GVR: 0.0000
  ✓ Best saved: MPJPE@560ms=112.6mm
Epoch  25/50 | loss: 0.00688 (recon: 0.00686, grav: 0.00224) | MPJPE@560ms: 101.5mm | GVR: 0.0000
  ✓ Best saved: MPJPE@560ms=101.5mm
Epoch  30/50 | loss: 0.00627 (recon: 0.00624, grav: 0.00209) | MPJPE@560ms: 111.5mm | GVR: 0.0000
Epoch  35/50 | loss: 0.00534 (recon: 0.00532, grav: 0.00191) | MPJPE@560ms: 104.6mm | GVR: 0.0000
Epoch  40/50 | loss: 0.00504 (recon: 0.00503, grav: 0.00184) | MPJPE@560ms: 108.2mm | GVR: 0.0000
Epoch  45/50 | loss: 0.00480 (recon: 0.00479, 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')